# Mass Spectrometry Analysis of Extracellular Vesicles Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading

We'll use `mlcroissant` and `pandas` to load metadata and records from the dataset. The main entry point is the provided Croissant JSON-LD URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '<No Name>')}: {getattr(metadata, 'description', '<No Description>')}")

## 2. Data Overview

We display all available record sets and their corresponding `@id`s. Listing record sets, fields, and column `@id`s is a good practice for programmatic exploration, especially when working with complex Croissant schemas. **All references will use the `@id` as recommended.**

In [ ]:
# Overview: List all record sets in the dataset
from collections import defaultdict

print('Available Record Sets:')
record_sets = list(dataset.metadata.record_sets)
record_sets_ids = []
for rset in record_sets:
    print(f"- Name: {getattr(rset, 'name', '<No name>')} | @id: {rset.id}")
    record_sets_ids.append(rset.id)

print('\n---')
# List fields for each record set
for rset in record_sets:
    print(f"Record Set: {getattr(rset, 'name', '<No name>')} | @id: {rset.id}")
    if hasattr(rset, 'fields'):
        for field in rset.fields:
            print(f"  Field: {getattr(field, 'name', '<No name>')} | @id: {field.id}")
            # If columns exist on field, show their @id
            if hasattr(field, 'columns'):
                for col in field.columns:
                    print(f"    Column: {getattr(col, 'name', '<No name>')} | @id: {col.id}")
    print('')

## 3. Data Extraction

Extract records from a chosen record set using its `@id`. You may adjust the specific record set or field by inserting its `@id` based on the output above. Multiple record sets can be loaded into separate DataFrames for analysis.

In [ ]:
# Example: Load all main record sets into DataFrames by their @id
# Please update 'main_record_set_id' and the list as needed based on the overview above.

# If the dataset contains only one main record set, select it for demonstration.
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]  # Example: use the first one
else:
    main_record_set_id = None

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading record set: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records_list = list(records_iter)
    if records_list:
        df = pd.DataFrame(records_list)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {record_set_id}")

# For demonstration, use the first DataFrame
df_main = dataframes[main_record_set_id] if main_record_set_id in dataframes else None

## 4. Exploratory Data Analysis (EDA)

We now perform some data processing and exploration steps using the loaded DataFrame. **Fields and columns are referenced via their `@id`**. Adjust the `numeric_field_id` and `group_field_id` as needed to match your specific dataset—use the IDs from Section 2.

Typical steps include:
 - Filtering records based on a numeric field
 - Normalizing the numeric field
 - Grouping by a category field (if present)

In [ ]:
# Find some likely numeric and category fields from the main record set
if df_main is not None:
    # Try to auto-detect numeric fields (float or int)
    numeric_fields = [col for col in df_main.columns if pd.api.types.is_numeric_dtype(df_main[col])]
    print(f"Numeric fields: {numeric_fields}")
    # Fallback: use manually identified @id if none are detected
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = df_main.columns[0]  # fallback to first column

    # Try to find a likely group field (object/string with low cardinality)
    candidate_group_fields = [col for col in df_main.columns 
                              if df_main[col].dtype == object and df_main[col].nunique() < 10]
    group_field_id = candidate_group_fields[0] if candidate_group_fields else df_main.columns[0]

    print(f"Selected numeric field for analysis (@id): {numeric_field_id}")
    print(f"Selected group field for grouping (@id): {group_field_id}")

    threshold = df_main[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df_main[numeric_field_id]) else 0
    # Filter records for demonstration
    try:
        filtered_df = df_main[df_main[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")
        print(filtered_df[[numeric_field_id, group_field_id]].head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    except Exception as e:
        print(f"Error during EDA: {e}")
else:
    print("No data available for EDA.")

## 5. Visualization

Let's plot the distribution of the selected numeric field (referenced by `@id`), and if possible, show a box plot grouped by a category field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_main is not None and numeric_field_id in df_main.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df_main[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Optionally, grouped boxplot if group_field is low-cardinality
    if group_field_id in df_main.columns and df_main[group_field_id].nunique() < 15:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_main)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()


## 6. Conclusion

In this notebook, we've used the `mlcroissant` library to explore the FAIR² dataset on mass spectrometry analysis of extracellular vesicles from human mast cells.

**Key exploration steps included:**
- Loading Croissant metadata (automatically detecting available record sets and field `@id`s)
- Loading tabular records into pandas DataFrames
- Programmatic selection and analysis of representative numeric and grouping fields
- Basic filtering, normalization, and summary group statistics
- Visualization of value distributions

See the [dataset on sen.science](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for further documentation and context, and extend this notebook for deeper analysis or integration into ML workflows.
